In [0]:
# =============================================================================
# HOSPITAL DATA PIPELINE — GOLD LAYER
# Description : Read Silver tables → join, aggregate, build business-ready
#               Gold tables (fact + dimension + summary marts).
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── Configuration ─────────────────────────────────────────────────────────────
SILVER_BASE_PATH = "abfss://silver@hospitaldatalake1.dfs.core.windows.net/Hospital_data/"
GOLD_BASE_PATH   = "abfss://gold@hospitaldatalake1.dfs.core.windows.net/Hospital_data/"

# ── Helper: write a Gold Delta table ─────────────────────────────────────────
def write_gold(df, table_name: str, description: str):
    path = f"{GOLD_BASE_PATH}{table_name}"
    df_out = (
        df
        .withColumn("_gold_ts",        F.current_timestamp())
        .withColumn("_pipeline_layer", F.lit("GOLD"))
        .withColumn("_batch_date",     F.current_date())
    )
    row_count = df_out.count()
    df_out.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)
    print(f"[GOLD] {description} → {table_name}  (rows: {row_count})")

# ── Load Silver tables from ADLS ─────────────────────────────────────────────
appointments = spark.read.format("delta").load(f"{SILVER_BASE_PATH}appointments")
billing      = spark.read.format("delta").load(f"{SILVER_BASE_PATH}billing")
treatments   = spark.read.format("delta").load(f"{SILVER_BASE_PATH}treatments")

# Derive full_name and age in gold (safe fallback if silver lacks these columns)
patients = (
    spark.read.format("delta").load(f"{SILVER_BASE_PATH}patients")
    .withColumn("full_name", F.concat_ws(" ", "first_name", "last_name"))
    .withColumn("age", F.floor(
        F.datediff(F.current_date(), F.col("date_of_birth")) / 365.25
    ).cast("int"))
)

doctors = (
    spark.read.format("delta").load(f"{SILVER_BASE_PATH}doctors")
    .withColumn("full_name", F.concat_ws(" ", "first_name", "last_name"))
)

print("[GOLD] All Silver tables loaded successfully.")

[GOLD] All Silver tables loaded successfully.


In [0]:
# =============================================================================
# GOLD TABLE 1 : fact_appointments
# Full enriched appointment fact — joins all five tables
# =============================================================================
fact_appointments = (
    appointments
    .join(patients.select(
              "patient_id", "full_name", "gender", "age",
              "insurance_provider", "insurance_number"),
          on="patient_id", how="left")
    .join(doctors.select(
              "doctor_id", F.col("full_name").alias("doctor_name"),
              "specialization", "hospital_branch", "years_experience"),
          on="doctor_id", how="left")
    .join(treatments.select(
              "appointment_id", "treatment_id",
              "treatment_type", "description", "cost"),
          on="appointment_id", how="left")
    .join(billing.select(
              "treatment_id", "bill_id",
              "amount", "payment_method", "payment_status"),
          on="treatment_id", how="left")
    .select(
        "appointment_id",
        "patient_id",
        F.col("full_name").alias("patient_name"),
        "gender",
        "age",
        "insurance_provider",
        "insurance_number",
        "doctor_id",
        "doctor_name",
        "specialization",
        "hospital_branch",
        "years_experience",
        "appointment_date",
        "appointment_time",
        "reason_for_visit",
        "status",
        "treatment_id",
        "treatment_type",
        "description",
        "cost",
        "bill_id",
        "amount",
        "payment_method",
        "payment_status",
    )
)

write_gold(fact_appointments, "fact_appointments",
           "Enriched appointment fact table")

[GOLD] ✅ Enriched appointment fact table → fact_appointments  (rows: 200)


In [0]:
# =============================================================================
# GOLD TABLE 2 : dim_patients
# Patient dimension with latest appointment info
# =============================================================================
latest_appt = (
    appointments
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("patient_id")
              .orderBy(F.col("appointment_date").desc())))
    .filter(F.col("rn") == 1)
    .select(
        "patient_id",
        F.col("appointment_date").alias("last_appointment_date"),
        F.col("status").alias("last_appointment_status"),
    )
)

dim_patients = (
    patients
    .join(latest_appt, on="patient_id", how="left")
    .select(
        "patient_id", "full_name", "first_name", "last_name",
        "gender", "age", "date_of_birth",
        "contact_number", "email", "address",
        "registration_date",
        "insurance_provider", "insurance_number",
        "last_appointment_date", "last_appointment_status"
    )
)

write_gold(dim_patients, "dim_patients", "Patient dimension table")

[GOLD] ✅ Patient dimension table → dim_patients  (rows: 50)


In [0]:
# =============================================================================
# GOLD TABLE 3 : dim_doctors
# Doctor dimension enriched with appointment & revenue KPIs
# =============================================================================
doctor_kpis = (
    fact_appointments
    .groupBy("doctor_id")
    .agg(
        F.count("appointment_id").alias("total_appointments"),
        F.countDistinct("patient_id").alias("unique_patients"),
        F.sum(F.when(F.col("status") == "Completed", 1).otherwise(0))
         .alias("completed_appointments"),
        F.sum(F.when(F.col("status") == "No-show", 1).otherwise(0))
         .alias("no_show_count"),
        F.round(F.sum("amount"), 2).alias("total_revenue_generated"),
        F.round(F.avg("amount"), 2).alias("avg_bill_amount"),
    )
)

dim_doctors = (
    doctors
    .join(doctor_kpis, on="doctor_id", how="left")
    .select(
        "doctor_id", "full_name", "first_name", "last_name",
        "specialization", "hospital_branch",
        "years_experience", "email", "phone_number",
        "total_appointments", "unique_patients",
        "completed_appointments", "no_show_count",
        "total_revenue_generated", "avg_bill_amount"
    )
)

write_gold(dim_doctors, "dim_doctors", "Doctor dimension with KPIs")

[GOLD] ✅ Doctor dimension with KPIs → dim_doctors  (rows: 10)


In [0]:
# =============================================================================
# GOLD TABLE 4 : agg_revenue_by_month
# Monthly revenue summary by hospital branch and payment method
# =============================================================================
agg_revenue_by_month = (
    fact_appointments
    .filter(F.col("payment_status") == "Paid")
    .withColumn("year",  F.year("appointment_date"))
    .withColumn("month", F.month("appointment_date"))
    .withColumn("month_name", F.date_format("appointment_date", "MMMM"))
    .groupBy("year", "month", "month_name", "hospital_branch", "payment_method")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.round(F.sum("amount"),  2).alias("total_revenue"),
        F.round(F.avg("amount"),  2).alias("avg_revenue_per_bill"),
        F.round(F.min("amount"),  2).alias("min_bill"),
        F.round(F.max("amount"),  2).alias("max_bill"),
    )
    .orderBy("year", "month", "hospital_branch")
)

write_gold(agg_revenue_by_month, "agg_revenue_by_month",
           "Monthly revenue by branch & payment method")

[GOLD] ✅ Monthly revenue by branch & payment method → agg_revenue_by_month  (rows: 49)


In [0]:
# =============================================================================
# GOLD TABLE 5 : agg_doctor_performance
# Doctor-level performance scorecard
# =============================================================================
agg_doctor_performance = (
    fact_appointments
    .groupBy("doctor_id", "doctor_name", "specialization", "hospital_branch")
    .agg(
        F.count("appointment_id").alias("total_appointments"),
        F.sum(F.when(F.col("status") == "Completed", 1).otherwise(0))
         .alias("completed"),
        F.sum(F.when(F.col("status") == "Cancelled", 1).otherwise(0))
         .alias("cancelled"),
        F.sum(F.when(F.col("status") == "No-show", 1).otherwise(0))
         .alias("no_shows"),
        F.sum(F.when(F.col("status") == "Scheduled", 1).otherwise(0))
         .alias("scheduled"),
        F.round(F.sum("amount"),  2).alias("total_revenue"),
        F.round(F.avg("cost"),    2).alias("avg_treatment_cost"),
        F.countDistinct("treatment_type").alias("treatment_types_handled"),
    )
    .withColumn(
        "completion_rate_pct",
        F.round(F.col("completed") / F.col("total_appointments") * 100, 2)
    )
    .withColumn(
        "no_show_rate_pct",
        F.round(F.col("no_shows") / F.col("total_appointments") * 100, 2)
    )
    .orderBy(F.col("total_revenue").desc())
)

write_gold(agg_doctor_performance, "agg_doctor_performance",
           "Doctor performance scorecard")

[GOLD] ✅ Doctor performance scorecard → agg_doctor_performance  (rows: 10)


In [0]:
# =============================================================================
# GOLD TABLE 6 : agg_treatment_summary
# Treatment type breakdown with cost and billing stats
# =============================================================================
agg_treatment_summary = (
    fact_appointments
    .groupBy("treatment_type")
    .agg(
        F.count("treatment_id").alias("total_treatments"),
        F.round(F.sum("cost"),      2).alias("total_cost"),
        F.round(F.avg("cost"),      2).alias("avg_cost"),
        F.round(F.min("cost"),      2).alias("min_cost"),
        F.round(F.max("cost"),      2).alias("max_cost"),
        F.round(F.sum("amount"),    2).alias("total_billed"),
        F.sum(F.when(F.col("payment_status") == "Paid",    1).otherwise(0))
         .alias("paid_count"),
        F.sum(F.when(F.col("payment_status") == "Pending", 1).otherwise(0))
         .alias("pending_count"),
        F.sum(F.when(F.col("payment_status") == "Failed",  1).otherwise(0))
         .alias("failed_count"),
    )
    .withColumn(
        "payment_success_rate_pct",
        F.round(F.col("paid_count") / F.col("total_treatments") * 100, 2)
    )
    .orderBy(F.col("total_treatments").desc())
)

write_gold(agg_treatment_summary, "agg_treatment_summary",
           "Treatment type summary with billing stats")

[GOLD] ✅ Treatment type summary with billing stats → agg_treatment_summary  (rows: 5)


In [0]:
# =============================================================================
# GOLD TABLE 7 : agg_patient_billing_summary
# Per-patient billing & visit overview (for patient portal / finance team)
# =============================================================================
agg_patient_billing_summary = (
    fact_appointments
    .groupBy("patient_id", "patient_name", "gender", "age",
             "insurance_provider")
    .agg(
        F.count("appointment_id").alias("total_visits"),
        F.round(F.sum("amount"),  2).alias("total_billed"),
        F.round(F.avg("amount"),  2).alias("avg_bill_per_visit"),
        F.sum(F.when(F.col("payment_status") == "Paid",    1).otherwise(0))
         .alias("paid_bills"),
        F.sum(F.when(F.col("payment_status") == "Pending", 1).otherwise(0))
         .alias("pending_bills"),
        F.sum(F.when(F.col("payment_status") == "Failed",  1).otherwise(0))
         .alias("failed_bills"),
        F.max("appointment_date").alias("last_visit_date"),
        F.min("appointment_date").alias("first_visit_date"),
    )
    .withColumn(
        "outstanding_amount",
        F.round(F.col("avg_bill_per_visit") * F.col("pending_bills"), 2)
    )
    .orderBy(F.col("total_billed").desc())
)

write_gold(agg_patient_billing_summary, "agg_patient_billing_summary",
           "Patient billing summary")

[GOLD] ✅ Patient billing summary → agg_patient_billing_summary  (rows: 48)


In [0]:
# =============================================================================
# ── Final summary ─────────────────────────────────────────────────────────────
# =============================================================================
print("\n" + "="*65)
print("  GOLD LAYER COMPLETE")
print("="*65)

gold_tables = [
    "fact_appointments",
    "dim_patients",
    "dim_doctors",
    "agg_revenue_by_month",
    "agg_doctor_performance",
    "agg_treatment_summary",
    "agg_patient_billing_summary",
]
for t in gold_tables:
    path = f"{GOLD_BASE_PATH}{t}"
    cnt = spark.read.format("delta").load(path).count()
    print(f"   {t:<35} rows: {cnt}")
print("="*65)


  GOLD LAYER COMPLETE
  ✅  fact_appointments                   rows: 200
  ✅  dim_patients                        rows: 50
  ✅  dim_doctors                         rows: 10
  ✅  agg_revenue_by_month                rows: 49
  ✅  agg_doctor_performance              rows: 10
  ✅  agg_treatment_summary               rows: 5
  ✅  agg_patient_billing_summary         rows: 48
